In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import probplot, mannwhitneyu
from IPython.display import display
%matplotlib inline

PATH_DATASET = "../data/raw/train.csv"
df = pd.read_csv(PATH_DATASET)
print("dimensiones del dataset:", df.shape)
display(df.head())

In [ ]:
def mannwhitneyu_method(df, m="", y="", onlyTrue=False):
    import pandas as pd
    import numpy as np
    from scipy.stats import mannwhitneyu

    useData = df.copy()
    #Creacion de variable indicadora
    nameBool = f"{m}_M"
    #Convertir la variable indicadora de ausencia a booleana
    useData[nameBool] = useData[m].isna()

    numeric_cols = useData.select_dtypes(include=[np.number]).columns.tolist()
    if "Id" in numeric_cols:
        numeric_cols.remove("Id")

    result = []

    for col in numeric_cols:
        groupWithOutMissing = useData.loc[useData[m].notna(), col].dropna()
        groupWithMissing = useData.loc[useData[m].isna(), col].dropna()

        if len(groupWithOutMissing) > 0 and len(groupWithMissing) > 0:
            stat, p_value = mannwhitneyu(
                groupWithMissing, groupWithOutMissing, alternative="two-sided"
            )
            result.append(
                {
                    "feature_Numerica": col,
                    "Estadistico_U": stat,
                    "p_value": p_value,
                    "Evidencia_MAR": p_value < 0.05,
                }
            )
    df_res = pd.DataFrame(result).sort_values(by="p_value")
    df_res['p_value'] = df_res['p_value'].round(5)
    if onlyTrue:
        df_res = df_res[df_res['Evidencia_MAR']]
    return df_res

In [ ]:
#PoolQC: Calidad de la piscina.
""" plt.figure(figsize=(10, 6))
sns.histplot(df['PoolArea'].dropna(), kde=True)
plt.show()

plt.figure(figsize=(10, 6))
sns.kdeplot(df['PoolArea'].dropna(), fill=True)
plt.show() """

resultado = mannwhitneyu_method(df, m="PoolQC", onlyTrue=True)
display(resultado)